# Finite elements in a nutshell

The Poisson equation as a model problem:

$$
-\Delta u = f \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial \Omega
$$

* $f$ a heat source, $u$ the temperature
* $f$ a charge density, $u$ the electrostatic potential
* replace $-\Delta$ by $-\frac{\hbar^2}{2m}\Delta + V$, or by $-\Delta - \omega^2 n^2$, and you are at the Schrödinger and Helmholtz equations of sections 4 and 5

## Weak formulation

Multiply by a test function $v$ with $v = 0$ on $\partial \Omega$, integrate over $\Omega$, and integrate by parts (Gauss' theorem):

$$
- \int_\Omega \Delta u \, v \, dx = \int_\Omega \nabla u \cdot \nabla v \, dx - \underbrace{\int_{\partial \Omega} \frac{\partial u}{\partial n} v \, ds}_{=0}
$$

Find $u \in H_0^1(\Omega)$ such that

$$
\underbrace{\int_\Omega \nabla u \cdot \nabla v \, dx}_{A(u,v)} = \underbrace{\int_\Omega f v \, dx}_{f(v)} \qquad \forall \, v \in H_0^1(\Omega)
$$

* only first derivatives are needed: functions with kinks are fine
* the natural space is the Sobolev space $H^1(\Omega) = \{ u \in L_2 : \nabla u \in L_2^d \}$
* $A(.,.)$ is a bilinear form, $f(.)$ a linear form

## Galerkin method

Replace $H_0^1$ by a finite dimensional subspace $V_h = \operatorname{span} \{ p_1, \ldots, p_N \}$, and expand $u_h = \sum_i u_i p_i$:

$$
\sum_{i=1}^N \underbrace{A(p_i, p_j)}_{A_{ji}} \, u_i = \underbrace{f(p_j)}_{f_j} \qquad j = 1, \ldots, N
$$

A linear system $A u = f$. Finite elements: the $p_i$ are piecewise polynomials on a mesh, each supported on a few elements only, so $A$ is **sparse**.

## Poisson equation in NGSolve

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

A mesh of the unit square, with mesh-size $h = 0.2$:

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))
Draw (mesh)
mesh.nv, mesh.ne

The finite element space: continuous, piecewise cubic polynomials, zero on the boundary:

In [ ]:
fes = H1(mesh, order=3, dirichlet=".*")
print ("number of dofs =", fes.ndof)

Bilinear form and linear form, written with trial and test functions, as on paper:

In [ ]:
u, v = fes.TnT()
a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f = LinearForm(50*x*y*v*dx).Assemble()

Solve the linear system on the free (non-Dirichlet) dofs:

In [ ]:
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(freedofs=fes.FreeDofs()) * f.vec
Draw (gfu);

In [ ]:
Draw (grad(gfu), mesh, order=3, vectors=True);

The sparse matrix, as a scipy matrix:

In [ ]:
import scipy.sparse as sp
A = sp.csr_matrix(a.mat.CSR())
print (A.shape, "nonzeros:", A.nnz, "=", A.nnz/A.shape[0], "per row")

## Basis functions

For `order=1`, the basis functions are the hat functions, one per vertex. Higher order adds edge and cell based basis functions.
Open the controls and use the `multidim` slider to walk through the vertex basis functions:

In [ ]:
gf = GridFunction(fes, multidim=mesh.nv)
for i in range (mesh.nv):
    gf.vecs[i][:] = 0
    gf.vecs[i][i] = 1
Draw (gf, mesh, deformation=True, animate=True, scale=0.3, euler_angles=[-65,-18,-45]);

basis functions associated with an edge:

In [ ]:
edgedofs = fes.GetDofNrs(NodeId(EDGE,40))
print ("dofs on edge #40: ", edgedofs)
for edof in edgedofs:
    gf.vec[:] = 0
    gf.vec[edof] = 1
    Draw (-3*gf, mesh, deformation=True, order=3)

## What we have seen

* weak form: integrate by parts, test with all $v$
* Galerkin: restrict to a finite element space, get a sparse system
* in NGSolve the code is the weak form: `grad(u)*grad(v)*dx`

The same recipe works for the PDEs of physics. What changes is the bilinear form, and the function space: $H^1$ for potentials and wave functions, $H(\operatorname{curl})$ for electric and magnetic fields, $H(\operatorname{div})$ for fluxes, and matrix-valued spaces for stresses and metrics.

More in the iFEM course: [jschoeberl.github.io/iFEM](https://jschoeberl.github.io/iFEM/intro.html)